# Прогноз цены авто — доработка

Ниже твой код. Я вставил **задания-подсказки** у проблемных мест. Исправь сам, потом покажи результат.

In [ ]:
import pandas as pd

# сюда загрузи свой датасет (как делал раньше)
df = pd.read_csv("cars.csv")
print(df.shape)
print(df.columns.tolist())

In [ ]:
# ======================================================================
# 🟠 ЗАДАНИЕ 3 — УТЕЧКА ДАННЫХ
# Ты заполняешь пропуски медианой ДО train_test_split:
#     X = X.fillna(X.median())
# Значит медиана считается по ВСЕМ данным, включая тест. Это подсматривание.
# Подумай: где правильнее заполнять пропуски, чтобы медиана считалась только по train?
# (Подсказка: SimpleImputer ВНУТРИ Pipeline.)
#
# 🟠 ЗАДАНИЕ 4 — МАЛО ПРИЗНАКОВ (главная причина низкого R²)
# Ты берёшь только 3 числовых признака. А где марка, топливо, коробка, кузов?
# Именно они сильнее всего влияют на цену. Посмотри df.columns и добавь категориальные.
# ======================================================================
df = df.dropna(subset=["price"])

X = df[[
    "year",
    "mileage",
    "engine_volume"
]]

y = df["price"]

X = X.fillna(X.median())

print("Готово!")
print("Признаки:", X.columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Обучение:", len(X_train))
print("Тест:", len(X_test))

In [ ]:
# ======================================================================
# 🔴 ЗАДАНИЕ 1 — BASELINE (обязательно по курсу)
# Прежде чем радоваться MAE, добавь "отметку на стене": DummyRegressor,
# который всегда предсказывает среднюю цену. С чем сравнивать модель, если
# нет самого простого варианта?  (Подсказка: from sklearn.dummy import DummyRegressor)
#
# 🔴 ЗАДАНИЕ 2 — PIPELINE (обязательно с урока 10)
# Модель обучается напрямую, без Pipeline. Заверни предобработку + модель в Pipeline.
# Для категориальных признаков понадобится ColumnTransformer + OneHotEncoder.
# ======================================================================
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Модель обучена!")

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

pred = model.predict(X_test)

# ⚠️ Сравни это число с MAE у baseline из Задания 1. Модель реально лучше?
print("Средняя ошибка:", round(mean_absolute_error(y_test, pred)), "сомони")
print("R²:", round(r2_score(y_test, pred), 3))

In [ ]:
# ⚠️ ЗАДАНИЕ 5 — ПРОВЕРЬ ВВОД
# mileage=500000 для авто 2015 года — это 500 тыс. км, нереально.
# Модель всё равно что-то выдаст. Попробуй реалистичные значения и сравни.
car = pd.DataFrame([{
    "year": 2015,
    "mileage": 500000,
    "engine_volume": 2.5
}])

price = model.predict(car)[0]

print("Предсказанная цена:", round(price), "сомони")

In [ ]:
# 🟡 ЗАДАНИЕ 6 — ВЕБ-ИНТЕРФЕЙС + README
# По финальной рубрике нужен интерфейс (Gradio) и README.
# Сделай простое окно, где вводят год/пробег/объём (и категории) и получают цену.